In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [ ]:
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9122 - loss: 0.2988 - val_accuracy: 0.9685 - val_loss: 0.1053
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9564 - loss: 0.1432 - val_accuracy: 0.9732 - val_loss: 0.0875
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9660 - loss: 0.1108 - val_accuracy: 0.9752 - val_loss: 0.0816
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9712 - loss: 0.0924 - val_accuracy: 0.9778 - val_loss: 0.0742
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9744 - loss: 0.0805 - val_accuracy: 0.9812 - val_loss: 0.0697
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9772 - loss: 0.0703 - val_accuracy: 0.9800 - val_loss: 0.0726
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9785 - loss: 0.0655 - val_accuracy: 0.9787 - val_loss: 0.0794
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9815 - loss: 0.0556 - 

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print(f"Test Accuracy: {test_acc:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9777 - loss: 0.0746
Test Accuracy: 0.9777


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint

In [ ]:

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

In [ ]:

dt_classifier = DecisionTreeClassifier()

param_grid = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy', 'log_loss']
}

grid_search = GridSearchCV(dt_classifier, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train_flat, y_train)

print("GridSearchCV - Top 3 combinations:")
results = grid_search.cv_results_
for i in range(3):
    print(f"Rank {results['rank_test_score'][i]}: Score = {results['mean_test_score'][i]:.4f}, Params = {results['params'][i]}")

Fitting 3 folds for each of 36 candidates, totalling 108 fits
GridSearchCV - Top 3 combinations:
Rank 34: Score = 0.6661, Params = {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 2}
Rank 34: Score = 0.6661, Params = {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 5}
Rank 34: Score = 0.6661, Params = {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 10}


In [ ]:

param_dist = {
    'max_depth': randint(low=5, high=25),
    'min_samples_split': randint(low=2, high=20),
    'criterion': ['gini', 'entropy', 'log_loss']
}

random_search = RandomizedSearchCV(dt_classifier, param_distributions=param_dist, n_iter=30, cv=3, scoring='accuracy', n_jobs=-1, verbose=1, random_state=42)
random_search.fit(X_train_flat, y_train)

print("\nRandomizedSearchCV - Best parameters and score:")
print(f"Best Score: {random_search.best_score_:.4f}")
print(f"Best Parameters: {random_search.best_params_}")

Fitting 3 folds for each of 30 candidates, totalling 90 fits

RandomizedSearchCV - Best parameters and score:
Best Score: 0.8707
Best Parameters: {'criterion': 'entropy', 'max_depth': 13, 'min_samples_split': 3}


In [ ]:
print("\n--- Comparison ---")


print("\nGridSearchCV Results:")
print(f"Best Score: {grid_search.best_score_:.4f}")
print(f"Best Parameters: {grid_search.best_params_}")

num_combinations_grid = len(param_grid['max_depth']) * len(param_grid['min_samples_split']) * len(param_grid['criterion'])
total_fits_grid = num_combinations_grid * grid_search.cv
print(f"Total Fits (GridSearchCV): {total_fits_grid}")

print("\nRandomizedSearchCV Results:")
print(f"Best Score: {random_search.best_score_:.4f}")
print(f"Best Parameters: {random_search.best_params_}")

total_fits_random = random_search.n_iter * random_search.cv
print(f"Total Fits (RandomizedSearchCV): {total_fits_random}")


--- Comparison ---

GridSearchCV Results:
Best Score: 0.8706
Best Parameters: {'criterion': 'log_loss', 'max_depth': 15, 'min_samples_split': 5}
Total Fits (GridSearchCV): 108

RandomizedSearchCV Results:
Best Score: 0.8707
Best Parameters: {'criterion': 'entropy', 'max_depth': 13, 'min_samples_split': 3}
Total Fits (RandomizedSearchCV): 90
